In [2]:
import pandas as pd
import sqlite3

# 1. Connect to SQLite database
conn = sqlite3.connect("data/database/bank_sqlite.db")

# 2. Load all tables into DataFrames
customers = pd.read_sql("SELECT * FROM customers;", conn)
accounts = pd.read_sql("SELECT * FROM accounts;", conn)
transactions = pd.read_sql("SELECT * FROM transactions;", conn)
loans = pd.read_sql("SELECT * FROM loans;", conn)
cards = pd.read_sql("SELECT * FROM cards;", conn)
branches = pd.read_sql("SELECT * FROM branches;", conn)
merchants = pd.read_sql("SELECT * FROM merchants;", conn)

print("Tables loaded successfully!")
print(
    f"Customers: {customers.shape}, Accounts: {accounts.shape}, Transactions:"
    f" {transactions.shape}"
)


Tables loaded successfully!
Customers: (50000, 7), Accounts: (75000, 5), Transactions: (1000000, 5)


In [3]:
customers["created_at"] = pd.to_datetime(customers["created_at"])
accounts["open_date"] = pd.to_datetime(accounts["open_date"])
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"]
)
loans["start_date"] = pd.to_datetime(loans["start_date"])
cards["expiration_date"] = pd.to_datetime(cards["expiration_date"])

# 2. Fill missing values in branches table
branches["city"] = branches["city"].fillna("Unknown")
branches["country"] = branches["country"].fillna("Unknown")

# 3. Data Integrity & Validation Checks
print(
    "Credit score range:"
    f" {customers['credit_score'].min()} - {customers['credit_score'].max()}"
)

orphaned_accounts = accounts[
    ~accounts["customer_id"].isin(customers["customer_id"])
]
print(f"Orphaned accounts found: {len(orphaned_accounts)}")

print("Preprocessing complete!")


Credit score range: 300 - 850
Orphaned accounts found: 0
Preprocessing complete!


In [5]:
dataframes = {
    "customers": customers,
    "accounts": accounts,
    "transactions": transactions,
    "loans": loans,
    "cards": cards,
    "branches": branches,
    "merchants": merchants,
}

for name, df in dataframes.items():
  print(f"=== {name.upper()} DATA TYPES ===")
  print(df.dtypes)
  print("-" * 30)


=== CUSTOMERS DATA TYPES ===
customer_id                str
first_name                 str
last_name                  str
email                      str
city                       str
credit_score             int64
created_at      datetime64[us]
dtype: object
------------------------------
=== ACCOUNTS DATA TYPES ===
account_id                 str
customer_id                str
account_type               str
balance_usd            float64
open_date       datetime64[us]
dtype: object
------------------------------
=== TRANSACTIONS DATA TYPES ===
transaction_id                 str
account_id                     str
merchant_id                    str
amount_usd                 float64
transaction_date    datetime64[us]
dtype: object
------------------------------
=== LOANS DATA TYPES ===
loan_id                     str
customer_id                 str
loan_amount             float64
interest_rate           float64
start_date       datetime64[us]
dtype: object
------------------------------

In [6]:
account_txns = (
    transactions.groupby("account_id")
    .agg(
        total_spent=("amount_usd", "sum"),
        avg_txn_amount=("amount_usd", "mean"),
        txn_count=("transaction_id", "count"),
    )
    .reset_index()
)

In [9]:
print("Raw Transactions Total:", transactions["amount_usd"].sum())
print("Aggregated Accounts Total:", account_txns["total_spent"].sum())


Raw Transactions Total: 5001164534.0
Aggregated Accounts Total: 5001164534.0


In [10]:
print("Unique Accounts in Transactions:", transactions["account_id"].nunique())
print("Rows in account_txns:", len(account_txns))


Unique Accounts in Transactions: 75000
Rows in account_txns: 75000


In [11]:
account_txns.describe()


,total_spent,avg_txn_amount,txn_count
count,75000.000000,75000.000000,75000.000000
mean,66682.193787,5000.993354,13.333333
std,21172.649426,827.610049,3.670924
min,2579.630000,1281.335000,1.000000
25%,51758.662500,4451.978591,11.000000
50%,65385.435000,5003.719639,13.000000
75%,80270.437500,5550.201810,16.000000
max,186975.710000,8689.030000,33.000000


In [12]:
accounts_summary = pd.merge(accounts, account_txns, on="account_id", how="left")
accounts_summary["total_spent"] = accounts_summary["total_spent"].fillna(0)


In [14]:
print("Original Accounts count:", len(accounts))
print("Merged accounts_summary count:", len(accounts_summary))


Original Accounts count: 75000
Merged accounts_summary count: 75000


In [15]:
print("Null count in total_spent:", accounts_summary["total_spent"].isnull().sum())


Null count in total_spent: 0


In [13]:
customer_profile = customers.merge(
    accounts_summary.groupby("customer_id")
    .agg(
        total_balance=("balance_usd", "sum"),
        total_spent=("total_spent", "sum"),
        total_accounts=("account_id", "count"),
    )
    .reset_index(),
    on="customer_id",
    how="left",
)

In [16]:
print("Original Customers count:", len(customers))
print("Customer Profile count:", len(customer_profile))


Original Customers count: 50000
Customer Profile count: 50000


In [17]:
print("Original Accounts Total Balance:", accounts["balance_usd"].sum())
print("Customer Profile Total Balance:", customer_profile["total_balance"].sum())
print("---")
print("Original Transactions Total Spend:", transactions["amount_usd"].sum())
print("Customer Profile Total Spend:", customer_profile["total_spent"].sum())


Original Accounts Total Balance: 7494138742.7699995
Customer Profile Total Balance: 7494138742.77
---
Original Transactions Total Spend: 5001164534.0
Customer Profile Total Spend: 5001164534.0


In [18]:
customer_profile[["total_balance", "total_spent", "total_accounts"]].isnull().sum()


total_balance     11151
total_spent       11151
total_accounts    11151
dtype: int64

In [20]:
# Fill 0 for customers without accounts
customer_profile[["total_balance", "total_spent", "total_accounts"]] = (
    customer_profile[
        ["total_balance", "total_spent", "total_accounts"]
    ].fillna(0)
)
print("Customer Profile DataFrame Created:")
customer_profile.head()

Customer Profile DataFrame Created:


,customer_id,first_name,last_name,email,city,credit_score,created_at,total_balance,total_spent,total_accounts
0,CUS000MKX5RHTAP,Abigail,Ashley,ruthwilliams@example.com,South Christopherton,827,2025-12-30 00:22:11,151677.32,26890.44,1.0
1,CUS002V4AVJO5UQ,Ralph,Obrien,brooke20@example.com,North Michaelport,510,2019-09-13 07:46:29,178960.28,76937.17,1.0
2,CUS004THQ8NDQW3,Andres,Stevens,robertsbenjamin@example.com,Port Faithstad,636,2024-01-01 18:57:58,131026.41,33025.48,1.0
3,CUS007GCM2J726A,Jessica,Davis,williamshaley@example.org,Lake Blaketown,492,2024-04-29 23:10:15,81689.89,107590.15,2.0
4,CUS00AO13A3Q5FO,Nicole,Murray,mendozajoshua@example.net,Sheriside,686,2019-05-27 09:28:46,189090.18,120446.58,2.0


In [21]:
# Aggregate loans by customer
customer_loans = (
    loans.groupby("customer_id")
    .agg(
        total_loan_amount=("loan_amount", "sum"),
        avg_interest_rate=("interest_rate", "mean"),
        loan_count=("loan_id", "count"),
    )
    .reset_index()
)

# Merge into customer_profile
customer_profile = customer_profile.merge(
    customer_loans, on="customer_id", how="left"
)
customer_profile[
    ["total_loan_amount", "avg_interest_rate", "loan_count"]
] = customer_profile[
    ["total_loan_amount", "avg_interest_rate", "loan_count"]
].fillna(
    0
)


In [24]:
customer_profile.dtypes

customer_id                     str
first_name                      str
last_name                       str
email                           str
city                            str
credit_score                  int64
created_at           datetime64[us]
total_balance               float64
total_spent                 float64
total_accounts              float64
total_loan_amount           float64
avg_interest_rate           float64
loan_count                  float64
dtype: object

In [25]:
# Calculate customer tenure (in days relative to max created_at date)
max_date = customers["created_at"].max()
customer_profile["tenure_days"] = (
    max_date - customer_profile["created_at"]
).dt.days

# Calculate last transaction date per customer
customer_last_txn = (
    transactions.merge(accounts[["account_id", "customer_id"]], on="account_id")
    .groupby("customer_id")["transaction_date"]
    .max()
    .reset_index()
    .rename(columns={"transaction_date": "last_txn_date"})
)

customer_profile = customer_profile.merge(
    customer_last_txn, on="customer_id", how="left"
)


In [26]:
account_type_flags = (
    pd.crosstab(accounts["customer_id"], accounts["account_type"])
    .add_prefix("has_")
    .reset_index()
)
customer_profile = customer_profile.merge(
    account_type_flags, on="customer_id", how="left"
).fillna(0)


In [27]:
customer_profile.dtypes

customer_id                     str
first_name                      str
last_name                       str
email                           str
city                            str
credit_score                  int64
created_at           datetime64[us]
total_balance               float64
total_spent                 float64
total_accounts              float64
total_loan_amount           float64
avg_interest_rate           float64
loan_count                  float64
tenure_days                   int64
last_txn_date                object
has_Business                float64
has_Checking                float64
has_Savings                 float64
dtype: object

In [31]:
flag_cols = [c for c in customer_profile.columns if c.startswith("has_")]
customer_profile = customer_profile.drop(columns=flag_cols)
print("Remaining columns:", customer_profile.columns.tolist())


Remaining columns: ['customer_id', 'first_name', 'last_name', 'email', 'city', 'credit_score', 'created_at', 'total_balance', 'total_spent', 'total_accounts', 'total_loan_amount', 'avg_interest_rate', 'loan_count', 'tenure_days', 'last_txn_date']


In [32]:
account_type_flags = (
    pd.crosstab(accounts["customer_id"], accounts["account_type"])
    .gt(0)        # Any count > 0 → True, else False
    .astype(int)  # True → 1, False → 0
    .add_prefix("has_")
    .reset_index()
)


In [33]:
customer_profile = customer_profile.merge(
    account_type_flags, on="customer_id", how="left"
)

flag_cols = [c for c in customer_profile.columns if c.startswith("has_")]
customer_profile[flag_cols] = customer_profile[flag_cols].fillna(0).astype(int)


In [35]:
customer_profile[flag_cols].head(10)



,has_Business,has_Checking,has_Savings
0,0,0,1
1,0,1,0
2,1,0,0
3,0,1,0
4,1,1,0
5,0,0,0
6,0,0,1
7,1,1,0
8,0,0,1
9,0,0,0


In [36]:
# Check orphaned cards (cards with no matching account)
orphaned_cards = cards[~cards["account_id"].isin(accounts["account_id"])]
print(f"Orphaned cards: {len(orphaned_cards)}")

# Check orphaned loans (loans with no matching customer)
orphaned_loans = loans[~loans["customer_id"].isin(customers["customer_id"])]
print(f"Orphaned loans: {len(orphaned_loans)}")

# Check orphaned transactions (transactions with no matching account)
orphaned_txns = transactions[~transactions["account_id"].isin(accounts["account_id"])]
print(f"Orphaned transactions: {len(orphaned_txns)}")

# Check expired cards
today = pd.Timestamp.now()
expired_cards = cards[cards["expiration_date"] < today]
print(f"Expired cards: {len(expired_cards)} out of {len(cards)}")


Orphaned cards: 0
Orphaned loans: 0
Orphaned transactions: 0
Expired cards: 19638 out of 100000


In [38]:
print(customer_profile.columns.tolist())


['customer_id', 'first_name', 'last_name', 'email', 'city', 'credit_score', 'created_at', 'total_balance', 'total_spent', 'total_accounts', 'total_loan_amount_x', 'avg_interest_rate_x', 'loan_count_x', 'tenure_days', 'last_txn_date', 'has_Business', 'has_Checking', 'has_Savings', 'total_loan_amount_y', 'avg_interest_rate_y', 'loan_count_y']


In [39]:
# Drop old loan columns if cell was already run before
old_loan_cols = ["total_loan_amount", "avg_interest_rate", "loan_count", "has_loan"]
existing = [c for c in old_loan_cols if c in customer_profile.columns]
customer_profile = customer_profile.drop(columns=existing)

# Aggregate loan stats per customer
customer_loans = (
    loans.groupby("customer_id")
    .agg(
        total_loan_amount=("loan_amount", "sum"),
        avg_interest_rate=("interest_rate", "mean"),
        loan_count=("loan_id", "count"),
    )
    .reset_index()
)

# Merge into customer_profile
customer_profile = customer_profile.merge(
    customer_loans, on="customer_id", how="left"
)

# Fill 0 for customers without loans
customer_profile[["total_loan_amount", "avg_interest_rate", "loan_count"]] = (
    customer_profile[["total_loan_amount", "avg_interest_rate", "loan_count"]].fillna(0)
)

# Binary flag
customer_profile["has_loan"] = (customer_profile["loan_count"] > 0).astype(int)

print("Customers with loans:", customer_profile["has_loan"].sum())


Customers with loans: 22540


In [40]:
# Join cards → accounts to get customer_id
cards_with_customer = cards.merge(
    accounts[["account_id", "customer_id"]], on="account_id", how="left"
)

# Pivot card types per customer
card_flags = (
    pd.crosstab(cards_with_customer["customer_id"], cards_with_customer["card_type"])
    .gt(0)
    .astype(int)
    .add_prefix("has_")
    .reset_index()
)
# Columns will be: has_Credit, has_Debit

customer_profile = customer_profile.merge(card_flags, on="customer_id", how="left")

# Fill 0 for customers with no cards
card_flag_cols = [c for c in customer_profile.columns if c.startswith("has_Credit") or c.startswith("has_Debit")]
customer_profile[card_flag_cols] = customer_profile[card_flag_cols].fillna(0).astype(int)

print("Card flag columns:", card_flag_cols)
customer_profile[["customer_id"] + card_flag_cols].head()


Card flag columns: ['has_Credit', 'has_Debit']


,customer_id,has_Credit,has_Debit
0,CUS000MKX5RHTAP,1,1
1,CUS002V4AVJO5UQ,1,0
2,CUS004THQ8NDQW3,0,1
3,CUS007GCM2J726A,1,1
4,CUS00AO13A3Q5FO,1,0


In [42]:
# Drop old columns if cell was already run before
old_cols = ["customer_tenure_days", "last_txn_date", "recency_days"]
customer_profile = customer_profile.drop(
    columns=[c for c in old_cols if c in customer_profile.columns]
)

# Customer tenure (days since created_at)
reference_date = pd.Timestamp.now()
customer_profile["customer_tenure_days"] = (
    reference_date - customer_profile["created_at"]
).dt.days

# Last transaction date per customer (via account_id → customer_id)
txn_with_customer = transactions.merge(
    accounts[["account_id", "customer_id"]], on="account_id", how="left"
)

customer_recency = (
    txn_with_customer.groupby("customer_id")["transaction_date"]
    .max()
    .reset_index()
    .rename(columns={"transaction_date": "last_txn_date"})
)

customer_profile = customer_profile.merge(
    customer_recency, on="customer_id", how="left"
)

# Days since last transaction (recency)
customer_profile["recency_days"] = (
    reference_date - customer_profile["last_txn_date"]
).dt.days

print("Tenure range (days):", customer_profile["customer_tenure_days"].min(),
      "–", customer_profile["customer_tenure_days"].max())
print("Recency range (days):", customer_profile["recency_days"].min(),
      "–", customer_profile["recency_days"].max())


Tenure range (days): 205 – 2762
Recency range (days): 205.0 – 2005.0


In [43]:
def credit_tier(score):
    if score < 580: return "Poor"
    elif score < 670: return "Fair"
    elif score < 740: return "Good"
    elif score < 800: return "Very Good"
    else: return "Exceptional"

customer_profile["credit_risk_tier"] = customer_profile["credit_score"].apply(credit_tier)
print(customer_profile["credit_risk_tier"].value_counts())


credit_risk_tier
Poor           25355
Fair            8163
Good            6453
Very Good       5378
Exceptional     4651
Name: count, dtype: int64


In [44]:
transactions["txn_year"] = transactions["transaction_date"].dt.year
transactions["txn_month"] = transactions["transaction_date"].dt.month
transactions["txn_day_of_week"] = transactions["transaction_date"].dt.day_name()
transactions["txn_hour"] = transactions["transaction_date"].dt.hour
transactions["txn_is_weekend"] = transactions["transaction_date"].dt.dayofweek.isin([5, 6]).astype(int)

transactions[["transaction_date", "txn_year", "txn_month", "txn_hour", "txn_is_weekend"]].head()


,transaction_date,txn_year,txn_month,txn_hour,txn_is_weekend
0,2022-01-14 04:58:46,2022,1,4,0
1,2021-02-20 00:12:15,2021,2,0,1
2,2024-06-02 11:20:37,2024,6,11,1
3,2020-03-08 22:36:55,2020,3,22,1
4,2024-09-01 06:49:37,2024,9,6,1


In [45]:
accounts["account_age_days"] = (reference_date - accounts["open_date"]).dt.days
accounts[["account_id", "account_type", "open_date", "account_age_days"]].head()


,account_id,account_type,open_date,account_age_days
0,ACC000NV80W3W25,Checking,2022-03-17 08:31:20,1591
1,ACC002FHD9SACSA,Checking,2021-10-28 12:55:25,1731
2,ACC002XR7B6XDZS,Business,2021-07-14 08:43:28,1837
3,ACC003O7MY9B1ZN,Checking,2025-01-19 12:33:15,552
4,ACC003R49JACSPQ,Checking,2019-03-16 11:05:54,2688


In [46]:
print("Shape:", customer_profile.shape)
print("Columns:", customer_profile.columns.tolist())
customer_profile.head()


Shape: (50000, 32)
Columns: ['customer_id', 'first_name', 'last_name', 'email', 'city', 'credit_score', 'created_at', 'total_balance', 'total_spent', 'total_accounts', 'total_loan_amount_x', 'avg_interest_rate_x', 'loan_count_x', 'tenure_days', 'last_txn_date_x', 'has_Business', 'has_Checking', 'has_Savings', 'total_loan_amount_y', 'avg_interest_rate_y', 'loan_count_y', 'total_loan_amount', 'avg_interest_rate', 'loan_count', 'has_loan', 'has_Credit', 'has_Debit', 'last_txn_date_y', 'customer_tenure_days', 'last_txn_date', 'recency_days', 'credit_risk_tier']


,customer_id,first_name,last_name,email,city,credit_score,created_at,total_balance,total_spent,total_accounts,...,avg_interest_rate,loan_count,has_loan,has_Credit,has_Debit,last_txn_date_y,customer_tenure_days,last_txn_date,recency_days,credit_risk_tier
0,CUS000MKX5RHTAP,Abigail,Ashley,ruthwilliams@example.com,South Christopherton,827,2025-12-30 00:22:11,151677.32,26890.44,1.0,...,0.00,0.0,0,1,1,2022-12-22 11:45:23,207,2022-12-22 11:45:23,1311.0,Exceptional
1,CUS002V4AVJO5UQ,Ralph,Obrien,brooke20@example.com,North Michaelport,510,2019-09-13 07:46:29,178960.28,76937.17,1.0,...,0.00,0.0,0,1,0,2025-10-24 15:12:07,2507,2025-10-24 15:12:07,273.0,Poor
2,CUS004THQ8NDQW3,Andres,Stevens,robertsbenjamin@example.com,Port Faithstad,636,2024-01-01 18:57:58,131026.41,33025.48,1.0,...,11.37,1.0,1,0,1,2025-01-08 13:13:47,935,2025-01-08 13:13:47,563.0,Fair
3,CUS007GCM2J726A,Jessica,Davis,williamshaley@example.org,Lake Blaketown,492,2024-04-29 23:10:15,81689.89,107590.15,2.0,...,0.00,0.0,0,1,1,2025-11-24 09:21:18,816,2025-11-24 09:21:18,243.0,Poor
4,CUS00AO13A3Q5FO,Nicole,Murray,mendozajoshua@example.net,Sheriside,686,2019-05-27 09:28:46,189090.18,120446.58,2.0,...,11.60,1.0,1,1,0,2025-12-07 10:25:40,2616,2025-12-07 10:25:40,230.0,Good


In [48]:
# Drop all stale duplicate columns
drop_cols = [c for c in customer_profile.columns if c.endswith("_x") or c.endswith("_y") or c == "tenure_days"]
customer_profile = customer_profile.drop(columns=drop_cols)

print("Clean shape:", customer_profile.shape)
print("Columns:", customer_profile.columns.tolist())


Clean shape: (50000, 23)
Columns: ['customer_id', 'first_name', 'last_name', 'email', 'city', 'credit_score', 'created_at', 'total_balance', 'total_spent', 'total_accounts', 'has_Business', 'has_Checking', 'has_Savings', 'total_loan_amount', 'avg_interest_rate', 'loan_count', 'has_loan', 'has_Credit', 'has_Debit', 'customer_tenure_days', 'last_txn_date', 'recency_days', 'credit_risk_tier']


In [51]:
# Save all enriched tables back to the database

# 1. customer_profile (new table - doesn't exist in DB yet)
customer_profile.to_sql("customer_profile", conn, if_exists="replace", index=False)
print("✅ Saved: customer_profile")

# 2. accounts_summary (new table - account + transaction aggregations)
accounts_summary.to_sql("accounts_summary", conn, if_exists="replace", index=False)
print("✅ Saved: accounts_summary")

# 3. accounts (enriched with account_age_days)
accounts.to_sql("accounts", conn, if_exists="replace", index=False)
print("✅ Saved: accounts (with account_age_days)")

# 4. transactions (enriched with time features)
transactions.to_sql("transactions", conn, if_exists="replace", index=False)
print("✅ Saved: transactions (with txn_year, txn_month, txn_hour, etc.)")

# 5. branches (filled missing city & country with 'Unknown')
branches.to_sql("branches", conn, if_exists="replace", index=False)
print("✅ Saved: branches (with filled city & country)")

# Verify: list all tables now in the database
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';", conn
)
print("\n📦 Tables now in database:")
print(tables["name"].tolist())


✅ Saved: customer_profile
✅ Saved: accounts_summary
✅ Saved: accounts (with account_age_days)
✅ Saved: transactions (with txn_year, txn_month, txn_hour, etc.)
✅ Saved: branches (with filled city & country)

📦 Tables now in database:
['cards', 'customers', 'loans', 'merchants', 'customer_profile', 'accounts_summary', 'accounts', 'transactions', 'branches']


In [52]:
import sqlite3, pandas as pd

conn = sqlite3.connect("data/database/bank_sqlite.db")

# List all tables
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';", conn
)["name"].tolist()

print(f"Tables in database: {tables}\n")

# Schema for each table
for table in tables:
    schema = pd.read_sql(f"PRAGMA table_info('{table}');", conn)
    print(f"\n{'='*40}")
    print(f"TABLE: {table.upper()}")
    print('='*40)
    print(schema[["cid", "name", "type", "notnull", "pk"]].to_string(index=False))


Tables in database: ['cards', 'customers', 'loans', 'merchants', 'customer_profile', 'accounts_summary', 'accounts', 'transactions', 'branches']


TABLE: CARDS
 cid            name      type  notnull  pk
   0         card_id      TEXT        0   0
   1      account_id      TEXT        0   0
   2       card_type      TEXT        0   0
   3 expiration_date TIMESTAMP        0   0

TABLE: CUSTOMERS
 cid         name      type  notnull  pk
   0  customer_id      TEXT        0   0
   1   first_name      TEXT        0   0
   2    last_name      TEXT        0   0
   3        email      TEXT        0   0
   4         city      TEXT        0   0
   5 credit_score   INTEGER        0   0
   6   created_at TIMESTAMP        0   0

TABLE: LOANS
 cid          name      type  notnull  pk
   0       loan_id      TEXT        0   0
   1   customer_id      TEXT        0   0
   2   loan_amount      REAL        0   0
   3 interest_rate      REAL        0   0
   4    start_date TIMESTAMP        0   0

TABLE: 